# Candle + egui: persistent live image window

This notebook opens a single egui window once and reuses it across cells.
You can send tensors to it to update the displayed image without spawning new windows.

Notes:
- On Linux this should work out-of-the-box. On macOS, GUI apps often must start on the main thread.
- We use a background thread to run the window, plus a channel for commands.
- Only one window instance stays open; subsequent calls just update the texture.

In [4]:
:dep candle-notebook = { path = "." }
:dep candle-eguiplot = { path = "eguiplot" }

// Optional runtime logging for troubleshooting (commented):
// :dep env_logger = "0.11"
// std::env::set_var("RUST_LOG", "info");
// env_logger::init();

In [7]:
use candle_notebook::candle::{Device, Tensor};
use candle_eguiplot as egui_plot;

// Convenience re-exports to demonstrate API usage:
pub use egui_plot::{show_tensor_gray as eguiplot_show_tensor_gray, show_tensor_rgb as eguiplot_show_tensor_rgb};

// Small helpers if needed later
fn make_checkerboard(h: usize, w: usize) -> Tensor {
    let mut v = vec![0f32; h*w];
    for y in 0..h { for x in 0..w { v[y*w + x] = if ((x/8 + y/8) % 2)==0 { 0.1 } else { 0.9 }; }}
    Tensor::from_vec(v, (h, w), &Device::Cpu).unwrap()
}

// Nothing else needed here; tests in later cells call the public APIs from the crate.

In [8]:
// Test 1: small checkerboard via the reusable crate
use candle_notebook::candle::{Tensor, Device};
use candle_eguiplot::show_tensor_gray as show_gray;

let dev = Device::Cpu;
let (h,w) = (128usize, 128usize);
let mut v = vec![0f32; h*w];
for y in 0..h { for x in 0..w { v[y*w + x] = if ((x/8 + y/8) % 2)==0 { 0.1 } else { 0.9 }; }}
let g = Tensor::from_vec(v, (h, w), &dev)?;
show_gray(&g)?;
println!("Sent checkerboard to egui window.");

Sent checkerboard to egui window.


In [9]:
// Test 2: dynamic updates (sine pattern for 60 frames) using candle-eguiplot
use candle_notebook::candle::{Tensor, Device};
use candle_eguiplot::show_tensor_gray as show_gray;

let dev = Device::Cpu;
let (h,w) = (256usize, 256usize);
for t in 0..60 {
    let mut v = vec![0f32; h*w];
    let ft = t as f32 * 0.1;
    for y in 0..h { for x in 0..w {
        let fx = x as f32 / 20.0; let fy = y as f32 / 24.0;
        v[y*w + x] = 0.5 + 0.5*( (fx+ft).sin() * (fy+ft*0.7).cos() );
    }}
    let g = Tensor::from_vec(v.clone(), (h, w), &dev)?;
    show_gray(&g)?;
    std::thread::sleep(std::time::Duration::from_millis(33));
}
println!("Sent 60 frames.");

Sent 60 frames.


In [11]:
// Test 3: RGB gradient using candle-eguiplot
use candle_notebook::candle::{Tensor, Device};
use candle_eguiplot::show_tensor_rgb as show_rgb;

let dev = Device::Cpu;
let (h,w) = (256usize, 256usize);
let mut f = vec![0f32; h*w*3];
for y in 0..h { for x in 0..w {
    let i = (y*w + x)*3;
    f[i+0] = x as f32 / (w as f32 - 1.0).max(1.0);
    f[i+1] = y as f32 / (h as f32 - 1.0).max(1.0);
    f[i+2] = ((x+y) as f32 * 0.5) / (((w+h) as f32) * 0.5).max(1.0);
}}
let rgb = Tensor::from_vec(f, (h, w, 3), &dev)?.permute((2,0,1))?; // (3,H,W)
show_rgb(&rgb)?;
println!("Sent RGB gradient.");

Sent RGB gradient.


In [ ]:
// Optional: clear stop flag before starting a new long-running loop
candle_eguiplot::clear_stop_flag();
println!("Stop flag cleared.");

Stop flag cleared.


Stop flag cleared.


In [15]:
// Test 4: loop until stop is requested (Q/Esc, Stop button, or closing the window)
use candle_notebook::candle::{Tensor, Device};
use candle_eguiplot::show_tensor_gray as show_gray;

let dev = Device::Cpu;
let (h,w) = (256usize, 256usize);
let mut frame: u64 = 0;
candle_eguiplot::clear_stop_flag();
println!("Starting loop. Press Q/Esc or click Stop in the egui window, or close it.");
loop {
    if candle_eguiplot::should_stop() {
        println!("Stop requested. Exiting loop at frame {frame}.");
        break;
    }
    let ft = frame as f32 * 0.08;
    let mut v = vec![0f32; h*w];
    for y in 0..h { for x in 0..w {
        let fx = x as f32 / 18.0; let fy = y as f32 / 22.0;
        v[y*w + x] = 0.5 + 0.5*( (fx+ft).sin() * (fy+ft*0.5).cos() );
    }}
    let g = Tensor::from_vec(v, (h, w), &dev)?;
    show_gray(&g)?;
    frame += 1;
    std::thread::sleep(std::time::Duration::from_millis(16));
}
println!("Loop ended.");

Starting loop. Press Q/Esc or click Stop in the egui window, or close it.
Stop requested. Exiting loop at frame 3905.
Loop ended.
